# Notebook 03 — Cluster Analysis

**Goal:** Identify user segments using KMeans and HDBSCAN on the behavioural feature matrix. Select the best model via silhouette/elbow analysis, evaluate cluster quality, and characterise each segment.

**Inputs:** `data/processed/user_features.parquet`

**Outputs:**
- `data/processed/cluster_labels.parquet` — user IDs with cluster assignments
- `outputs/figures/elbow.html` — KMeans model selection chart
- `outputs/figures/umap_clusters.html` — 2D UMAP scatter
- `outputs/figures/cluster_heatmap.html` — feature profile heatmap

In [ ]:
import sys
import logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

import pandas as pd
import numpy as np

Path('../outputs/figures').mkdir(parents=True, exist_ok=True)

In [ ]:
from src.data.loader import load_config

cfg = load_config('../configs/config.yaml')
cluster_cfg = cfg['clustering']

feature_matrix = pd.read_parquet('../data/processed/user_features.parquet')
print(f'Feature matrix: {feature_matrix.shape[0]} users × {feature_matrix.shape[1]} features')

## 1. KMeans — Elbow & Silhouette Analysis

Sweep k from 3 to 10. PCA is applied first to remove multicollinearity.

In [ ]:
from src.clustering.pipeline import run_clustering_pipeline
from src.clustering.evaluation import elbow_data
from src.visualization.plots import plot_elbow, save_figure

# Run KMeans sweep
kmeans_result = run_clustering_pipeline(
    feature_matrix=feature_matrix,
    algorithm='kmeans',
    use_pca=True,
    use_umap_viz=True,
    config=cluster_cfg,
)

print(f'Best k: {kmeans_result.n_clusters}')
print(f'Silhouette: {kmeans_result.silhouette:.4f}')
print(f'Davies-Bouldin: {kmeans_result.davies_bouldin:.4f}')
print(f'Calinski-Harabasz: {kmeans_result.calinski_harabasz:.1f}')

In [ ]:
# The pipeline already did the sweep internally;
# re-run sweep explicitly to capture all inertia/silhouette values for elbow plot
from src.clustering.pipeline import preprocess, reduce_with_pca, cluster_kmeans

X_scaled, feature_names = preprocess(feature_matrix)
X_pca, pca_obj = reduce_with_pca(X_scaled)

k_min = cluster_cfg['kmeans']['n_clusters_range'][0]
k_max = cluster_cfg['kmeans']['n_clusters_range'][1]

_, best_k, sil_scores, inertias = cluster_kmeans(
    X_pca,
    k_range=(k_min, k_max),
    n_init=cluster_cfg['kmeans']['n_init'],
    random_state=cluster_cfg['kmeans']['random_state'],
)

elbow_df = elbow_data(inertias, sil_scores)
fig_elbow = plot_elbow(elbow_df)
save_figure(fig_elbow, '../outputs/figures/elbow')
fig_elbow.show()

## 2. HDBSCAN — Density-Based Clustering

In [ ]:
hdbscan_result = run_clustering_pipeline(
    feature_matrix=feature_matrix,
    algorithm='hdbscan',
    use_pca=True,
    use_umap_viz=True,
    config=cluster_cfg,
)

print(f'HDBSCAN clusters: {hdbscan_result.n_clusters}')
print(f'Noise points: {(hdbscan_result.labels == -1).sum()}')
print(f'Silhouette: {hdbscan_result.silhouette:.4f}')

## 3. Model Selection

Compare KMeans (best k) vs HDBSCAN on key metrics.

In [ ]:
comparison = pd.DataFrame([
    {
        'Algorithm': f'KMeans (k={kmeans_result.n_clusters})',
        'Clusters': kmeans_result.n_clusters,
        'Noise Points': 0,
        'Silhouette ↑': kmeans_result.silhouette,
        'Davies-Bouldin ↓': kmeans_result.davies_bouldin,
        'Calinski-Harabasz ↑': kmeans_result.calinski_harabasz,
    },
    {
        'Algorithm': 'HDBSCAN',
        'Clusters': hdbscan_result.n_clusters,
        'Noise Points': int((hdbscan_result.labels == -1).sum()),
        'Silhouette ↑': hdbscan_result.silhouette,
        'Davies-Bouldin ↓': hdbscan_result.davies_bouldin,
        'Calinski-Harabasz ↑': hdbscan_result.calinski_harabasz,
    },
])
comparison.set_index('Algorithm', inplace=True)
comparison.round(4)

In [ ]:
# Model selection — choose the algorithm with the strongest evidence of
# well-separated, stable clusters using three complementary metrics.
#
# Silhouette score  (↑ better, range -1..1):
#   Measures how similar each point is to its own cluster versus the nearest
#   other cluster.  Values above 0.25 indicate meaningful separation.
#
# Davies-Bouldin index (↓ better, ≥0):
#   Ratio of within-cluster scatter to between-cluster distance.
#   Lower = more compact, well-separated clusters.
#
# Calinski-Harabasz index (↑ better, ≥0):
#   Ratio of between-cluster dispersion to within-cluster dispersion.
#   Higher = denser, more distinct clusters.
#
# Decision rule:
#   Silhouette is the primary criterion because it is bounded and directly
#   interpretable.  Davies-Bouldin and Calinski-Harabasz serve as tie-breakers
#   and sanity checks.  HDBSCAN is preferred over KMeans only if it produces a
#   higher silhouette AND fewer than 30 % noise points (unlabelled points
#   degrade downstream personalisation).

noise_fraction = (hdbscan_result.labels == -1).mean()
NOISE_THRESHOLD = 0.30

sil_gap   = kmeans_result.silhouette - hdbscan_result.silhouette
db_gap    = hdbscan_result.davies_bouldin - kmeans_result.davies_bouldin   # positive = KMeans better
ch_gap    = kmeans_result.calinski_harabasz - hdbscan_result.calinski_harabasz

print('=' * 65)
print('MODEL SELECTION SUMMARY')
print('=' * 65)
print(f'{"Metric":<35} {"KMeans":>10} {"HDBSCAN":>10}')
print('-' * 65)
print(f'{"Silhouette (↑ better)":<35} {kmeans_result.silhouette:>10.4f} {hdbscan_result.silhouette:>10.4f}')
print(f'{"Davies-Bouldin (↓ better)":<35} {kmeans_result.davies_bouldin:>10.4f} {hdbscan_result.davies_bouldin:>10.4f}')
print(f'{"Calinski-Harabasz (↑ better)":<35} {kmeans_result.calinski_harabasz:>10.1f} {hdbscan_result.calinski_harabasz:>10.1f}')
print(f'{"Noise fraction":<35} {"0.00%":>10} {noise_fraction * 100:>9.1f}%')
print('-' * 65)

# Selection logic — transparent and reproducible
if noise_fraction > NOISE_THRESHOLD:
    best_result = kmeans_result
    reason = (f'HDBSCAN noise fraction ({noise_fraction * 100:.1f} %) exceeds the '
              f'{NOISE_THRESHOLD * 100:.0f} % threshold — too many users unassigned '
              f'for reliable personalisation.  KMeans guarantees full coverage.')
elif kmeans_result.silhouette >= hdbscan_result.silhouette:
    best_result = kmeans_result
    reason = (f'KMeans silhouette ({kmeans_result.silhouette:.4f}) ≥ HDBSCAN '
              f'({hdbscan_result.silhouette:.4f}).  '
              + ('Davies-Bouldin also favours KMeans. ' if db_gap <= 0 else '')
              + f'Selecting KMeans k={kmeans_result.n_clusters}.')
else:
    best_result = hdbscan_result
    reason = (f'HDBSCAN silhouette ({hdbscan_result.silhouette:.4f}) > KMeans '
              f'({kmeans_result.silhouette:.4f}) and noise fraction '
              f'({noise_fraction * 100:.1f} %) is within acceptable range.  '
              f'Selecting HDBSCAN ({hdbscan_result.n_clusters} clusters).')

print(f'\nSELECTED: {best_result.algorithm.upper()}')
print(f'REASON   : {reason}')
print('=' * 65)

## 4. Cluster Stability (Bootstrap Silhouette)

In [ ]:
from src.clustering.evaluation import bootstrap_silhouette

mean_sil, std_sil = bootstrap_silhouette(
    best_result.feature_matrix_scaled,
    best_result.labels,
    n_bootstrap=50,
)
print(f'Bootstrap silhouette: {mean_sil:.4f} ± {std_sil:.4f}')

## 5. Cluster Profiling

In [ ]:
from src.clustering.evaluation import summarise_clusters, label_clusters, feature_importance

cluster_summary = summarise_clusters(feature_matrix, best_result.labels)
cluster_names = label_clusters(cluster_summary, feature_matrix, best_result.labels)

print('Cluster labels:')
for cid, name in cluster_names.items():
    n = (best_result.labels == cid).sum()
    print(f'  Cluster {cid} ({n} users): {name}')

In [ ]:
# Feature importance across clusters
imp_df = feature_importance(feature_matrix, best_result.labels)
print('Top 10 most discriminating features:')
print(imp_df.head(10).to_string(index=False))

## 6. Save Cluster Labels

In [ ]:
labels_df = pd.DataFrame({
    'userid': feature_matrix.index,
    'cluster': best_result.labels,
    'cluster_name': [cluster_names.get(c, str(c)) for c in best_result.labels],
})

if best_result.umap_coords is not None:
    labels_df['umap_x'] = best_result.umap_coords[:, 0]
    labels_df['umap_y'] = best_result.umap_coords[:, 1]

labels_df.to_parquet('../data/processed/cluster_labels.parquet', index=False)
print(f'Cluster labels saved: {labels_df.shape}')
labels_df['cluster_name'].value_counts()

Proceed to **Notebook 04** for interactive visualisations and business insights.